In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
print(RAW.resolve())
print([p.name for p in RAW.iterdir()])

D:\Documents\quantile-demand-forecasting-for-inventory-replenishment-policy-optimization\data\raw
['calendar.csv', 'sales_train_evaluation.csv', 'sales_train_validation.csv', 'sample_submission.csv', 'sell_prices.csv']


In [2]:
cal = pd.read_csv(RAW / "calendar.csv")
print(cal.shape)
cal.head()

(1969, 14)


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [3]:
prices = pd.read_csv(RAW / "sell_prices.csv")
print(prices.shape)
prices.head()

(6841121, 4)


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [4]:
sales_head = pd.read_csv(RAW / "sales_train_evaluation.csv", nrows=5)
print(sales_head.shape)
sales_head.iloc[:, :10]

(5, 1947)


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0


In [5]:
ids = pd.read_csv(
    RAW / "sales_train_evaluation.csv",
    usecols=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
)
print(ids.shape)
print(ids.nunique())
print(ids["cat_id"].unique())
print(ids["state_id"].unique())

(30490, 6)
id          30490
item_id      3049
dept_id         7
cat_id          3
store_id       10
state_id        3
dtype: int64
<ArrowStringArray>
['HOBBIES', 'HOUSEHOLD', 'FOODS']
Length: 3, dtype: str
<ArrowStringArray>
['CA', 'TX', 'WI']
Length: 3, dtype: str


In [6]:
cols = pd.read_csv(RAW / "sales_train_evaluation.csv", nrows=0).columns
day_cols = [c for c in cols if c.startswith("d_")]
print("số cột ngày:", len(day_cols))
print("mọi ngày đều có trong calendar:", set(day_cols) <= set(cal["d"]))

print("dòng giá bị trùng:", prices.duplicated(["store_id", "item_id", "wm_yr_wk"]).sum())

val_cols = pd.read_csv(RAW / "sales_train_validation.csv", nrows=0).columns
val_days = [c for c in val_cols if c.startswith("d_")]
print("validation có", len(val_days), "ngày; evaluation có", len(day_cols), "ngày")
print("validation là phần đầu của evaluation:", day_cols[:len(val_days)] == val_days)

số cột ngày: 1941
mọi ngày đều có trong calendar: True
dòng giá bị trùng: 0
validation có 1913 ngày; evaluation có 1941 ngày
validation là phần đầu của evaluation: True


In [6]:
import numpy as np

day_cols = [c for c in pd.read_csv(RAW / "sales_train_evaluation.csv", nrows=0).columns
            if c.startswith("d_")]
sales = pd.read_csv(RAW / "sales_train_evaluation.csv",
                    dtype={c: "int32" for c in day_cols})
print(sales.shape)

(30490, 1947)


In [7]:
X = sales[day_cols].to_numpy()        # ma trận: mỗi dòng là một chuỗi, mỗi cột là một ngày
pos = X > 0                           # True nếu hôm đó bán được

stats = sales[["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]].copy()
stats["mean_daily"] = X.mean(axis=1)                      # trung bình số bán mỗi ngày
launched = np.cumsum(pos, axis=1) > 0                     # True từ ngày bán đầu tiên trở đi
stats["days_live"] = launched.sum(axis=1)                 # số ngày kể từ lần bán đầu tiên
zeros_live = ((~pos) & launched).sum(axis=1)              # số ngày bán 0 sau khi đã bắt đầu bán
stats["zero_share_live"] = zeros_live / np.where(stats["days_live"] == 0, np.nan, stats["days_live"])
stats.head()

,id,item_id,dept_id,cat_id,store_id,state_id,mean_daily,days_live,zero_share_live
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0.326121,1040,0.580769
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0.257599,1798,0.773081
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0.159196,836,0.726077
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,1.719217,1905,0.309186
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0.972694,1829,0.462548


In [8]:
print(stats.groupby("cat_id")[["mean_daily", "zero_share_live"]].median())
print(stats.groupby("store_id")[["mean_daily", "zero_share_live"]].median())
print("tỉ lệ chuỗi bắt đầu bán muộn:", (stats["days_live"] < len(day_cols)).mean())
print(stats["days_live"].describe())

           mean_daily  zero_share_live
cat_id                                
FOODS        0.710459         0.538897
HOBBIES      0.234930         0.778092
HOUSEHOLD    0.329727         0.695540
          mean_daily  zero_share_live
store_id                             
CA_1        0.591963         0.579083
CA_2        0.458011         0.597796
CA_3        0.775374         0.504379
CA_4        0.340031         0.683505
TX_1        0.357548         0.696466
TX_2        0.477589         0.629057
TX_3        0.381762         0.675402
WI_1        0.446677         0.615584
WI_2        0.397218         0.664606
WI_3        0.374549         0.682638
tỉ lệ chuỗi bắt đầu bán muộn: 0.7711052804198097
count    30490.000000
mean      1534.805510
std        477.176658
min         96.000000
25%       1175.000000
50%       1782.000000
75%       1940.000000
max       1941.000000
Name: days_live, dtype: float64


In [9]:
base = stats[(stats["cat_id"] == "FOODS")
             & (stats["store_id"].isin(["CA_3", "TX_2", "WI_1"]))]
print("tổng chuỗi FOODS ở 3 cửa hàng:", len(base))
for t in [500, 1000, 1500, 1800]:
    print(t, "ngày:", (base["days_live"] >= t).sum(), "chuỗi")

tổng chuỗi FOODS ở 3 cửa hàng: 4311
500 ngày: 4236 chuỗi
1000 ngày: 3584 chuỗi
1500 ngày: 2472 chuỗi
1800 ngày: 2056 chuỗi


In [12]:
CAT = "FOODS"
STORES = ["CA_3", "TX_2", "WI_1"]
MIN_LIVE_DAYS = 1000
N_PER_CELL = 33       # mỗi (cửa hàng × mức bán) lấy 33 chuỗi 
SEED = 29

pool = stats[(stats["cat_id"] == CAT)
             & (stats["store_id"].isin(STORES))
             & (stats["days_live"] >= MIN_LIVE_DAYS)].copy()

# trung bình số bán mỗi ngày, chỉ tính từ lần bán đầu tiên
pool["mean_live"] = pool["mean_daily"] * len(day_cols) / pool["days_live"]

# chia 3 mức bán (0 = thấp, 1 = vừa, 2 = cao) riêng trong từng cửa hàng
pool["volume_group"] = pool.groupby("store_id")["mean_live"].transform(
    lambda s: pd.qcut(s, 3, labels=False))

parts = [g.sample(min(len(g), N_PER_CELL), random_state=SEED)
         for _, g in pool.groupby(["store_id", "volume_group"])]
subset = pd.concat(parts)

print("số chuỗi đủ điều kiện:", len(pool), "| số chuỗi chọn:", len(subset))
print(subset.groupby(["store_id", "volume_group"]).size())

số chuỗi đủ điều kiện: 3584 | số chuỗi chọn: 297
store_id  volume_group
CA_3      0               33
          1               33
          2               33
TX_2      0               33
          1               33
          2               33
WI_1      0               33
          1               33
          2               33
dtype: int64


In [14]:
Path("../configs").mkdir(exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

subset.to_csv("../configs/subset_ids.csv", index=False)       # danh sách chuỗi: nhỏ, nên đưa lên Git
sales[sales["id"].isin(subset["id"])].to_parquet(
    "../data/processed/sales_subset_wide.parquet", index=False)  # dữ liệu: lớn, không đưa lên Git